## Library Imports

In [3]:
import os
import time
import warnings
from collections import Counter
from functools import reduce
from itertools import combinations

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from matplotlib.colors import to_rgba

from sklearn.decomposition import PCA, FastICA, IncrementalPCA, KernelPCA
from sklearn.cross_decomposition import CCA
from sklearn.kernel_approximation import RBFSampler
from sklearn.manifold import Isomap, TSNE, SpectralEmbedding, LocallyLinearEmbedding
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer,
                                   QuantileTransformer, normalize)
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import (LogisticRegression, RidgeClassifier, SGDClassifier,
                                  PassiveAggressiveClassifier, Perceptron, LinearRegression,
                                  Ridge, Lasso, ElasticNet)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier,
                             RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor)
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                            roc_auc_score, cohen_kappa_score, balanced_accuracy_score,
                            r2_score, mean_absolute_error, mean_squared_error)
from sklearn.experimental import enable_iterative_imputer
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import confusion_matrix

## Dataset Loading and Preprocessing

In [4]:
dataset = "ppbr_az"
labels_df_file = f"../data/Benchmarking_Datasets/TDC/ADME/{dataset}.tab"
target_column_name = "Y"
labels_df = pd.read_csv(labels_df_file,sep="\t")
labels_df = labels_df.rename(columns={"Drug": "SMILES"})
labels_df

,Drug_ID,SMILES,Y,Species
0,CHEMBL1017,CCCc1nc2c(C)cc(-c3nc4ccccc4n3C)cc2n1Cc1ccc(-c2...,98.25,Canis lupus familiaris
1,CHEMBL2337981,CC(C)(C)NC(=O)NCCN1CCC(CO)(CNC(=O)c2cc(Cl)cc(C...,82.05,Canis lupus familiaris
2,CHEMBL401158,CCCCNc1nc(SCCC)nc2c1nnn2[C@@H]1C[C@H](CO)[C@@H...,95.33,Canis lupus familiaris
3,CHEMBL2337980,CC(C)(C)NC(=O)NCCN1CCC(O)(CNC(=O)c2cc(Cl)cc(Cl...,64.01,Canis lupus familiaris
4,CHEMBL914,CC(C)(C(=O)O)c1ccc(C(O)CCCN2CCC(C(O)(c3ccccc3)...,84.90,Canis lupus familiaris
...,...,...,...,...
2823,CHEMBL2088414,O=S(=O)(Nc1ccc2sc(CO)nc2c1)c1ccc(Br)cc1,99.82,Rattus norvegicus
2824,CHEMBL1628227,CN(C)CCC=C1c2ccccc2COc2ccccc21,79.55,Rattus norvegicus
2825,CHEMBL1287853,Cc1cnc(Nc2ccc(OCCN3CCCC3)cc2)nc1Nc1cccc(S(=O)(...,79.55,Rattus norvegicus
2826,CHEMBL1289283,O=c1cc(OCc2ccccc2)ccn1-c1ccc2c(cnn2CCN2CCCC2)c1,88.11,Rattus norvegicus


## Data Quality Checks

In [5]:
conflicting_smiles = labels_df.groupby('SMILES')[target_column_name].nunique()
conflicting_smiles = conflicting_smiles[conflicting_smiles > 1].index.tolist()

if conflicting_smiles:
    print(len(conflicting_smiles),"Conflicting SMILES found:")
    for smiles in conflicting_smiles:
        print(smiles)
        display(labels_df[labels_df['SMILES'] == smiles])
else:
    print("No conflicting SMILES found.")

conflicting_smiles = labels_df.groupby('SMILES')[target_column_name].nunique()

691 Conflicting SMILES found:
C#CC[C@@H](C(=O)N[C@H](/C=C/C(=O)OCC)C[C@@H]1CCNC1=O)n1cccc(NC(=O)c2cc(C)on2)c1=O


,Drug_ID,SMILES,Y,Species
50,CHEMBL141157,C#CC[C@@H](C(=O)N[C@H](/C=C/C(=O)OCC)C[C@@H]1C...,46.55,Canis lupus familiaris
1667,CHEMBL141157,C#CC[C@@H](C(=O)N[C@H](/C=C/C(=O)OCC)C[C@@H]1C...,55.73,Homo sapiens


C/C=C/c1ccc(Oc2nnnn2-c2ccc(C(N)=O)cc2)c(OC)c1


,Drug_ID,SMILES,Y,Species
575,CHEMBL1320158,C/C=C/c1ccc(Oc2nnnn2-c2ccc(C(N)=O)cc2)c(OC)c1,97.32,Homo sapiens
2071,CHEMBL1320158,C/C=C/c1ccc(Oc2nnnn2-c2ccc(C(N)=O)cc2)c(OC)c1,95.72,Mus musculus


C1Cc2[nH]nc(-c3nn[nH]n3)c2C1


...

,Drug_ID,SMILES,Y,Species
1940,CHEMBL510780,c1ccc2c(c1)CCC[C@H]2Nc1nc2ccccc2[nH]1,99.30,Homo sapiens
2324,CHEMBL510780,c1ccc2c(c1)CCC[C@H]2Nc1nc2ccccc2[nH]1,95.63,Rattus norvegicus


c1ccc2nc(COc3ccc(-c4n[nH]cc4-c4ccncc4)cc3)ccc2c1


,Drug_ID,SMILES,Y,Species
158,CHEMBL560377,c1ccc2nc(COc3ccc(-c4n[nH]cc4-c4ccncc4)cc3)ccc2c1,99.80,Canis lupus familiaris
870,CHEMBL560377,c1ccc2nc(COc3ccc(-c4n[nH]cc4-c4ccncc4)cc3)ccc2c1,99.80,Homo sapiens
1969,CHEMBL560377,c1ccc2nc(COc3ccc(-c4n[nH]cc4-c4ccncc4)cc3)ccc2c1,99.87,Mus musculus
2822,CHEMBL560377,c1ccc2nc(COc3ccc(-c4n[nH]cc4-c4ccncc4)cc3)ccc2c1,99.82,Rattus norvegicus


c1cncc(Nc2ncc(-c3ccncn3)c(-c3ccco3)n2)c1


,Drug_ID,SMILES,Y,Species
809,CHEMBL375293,c1cncc(Nc2ncc(-c3ccncn3)c(-c3ccco3)n2)c1,86.05,Homo sapiens
1950,CHEMBL375293,c1cncc(Nc2ncc(-c3ccncn3)c(-c3ccco3)n2)c1,84.00,Mus musculus
2806,CHEMBL375293,c1cncc(Nc2ncc(-c3ccncn3)c(-c3ccco3)n2)c1,86.05,Rattus norvegicus


In [6]:
conflicting_smiles = conflicting_smiles[conflicting_smiles > 1].index.tolist()

if conflicting_smiles:
    print("Removing conflicting SMILES:")
    labels_df_filtered = labels_df[~labels_df['SMILES'].isin(conflicting_smiles)]
    print("Done")
    display(labels_df_filtered)
    print("Conflicting SMILES removed")
else:
    labels_df_filtered = labels_df.copy()

Removing conflicting SMILES:
Done


,Drug_ID,SMILES,Y,Species
28,CHEMBL1922660,CCOc1ccc(-c2ccc(Cn3c(CC(C)(C)C(=O)O)c(SC(C)(C)...,99.92,Canis lupus familiaris
30,CHEMBL403225,C[C@H](CO)Nc1nc(SCc2cccc(Cl)c2F)nc2[nH]c(=O)sc12,99.95,Canis lupus familiaris
45,CHEMBL1790041,CNC(=C[N+](=O)[O-])NCCSCc1ccc(CN(C)C)o1,28.95,Canis lupus familiaris
62,CHEMBL2171021,Cc1c(Cl)ccc(OC2CCN(C3CCN(C(=O)c4cccc(S(C)(=O)=...,96.50,Canis lupus familiaris
92,CHEMBL1516,CCCc1nc(C(C)(C)O)c(C(=O)O)n1Cc1ccc(-c2ccccc2-c...,98.81,Canis lupus familiaris
...,...,...,...,...
2791,CHEMBL1580912,O=C(NCCc1ccc(Cl)cc1)c1cccnc1,91.46,Rattus norvegicus
2796,CHEMBL31354,COc1ccccc1N1CCN(CCN(C(=O)C2CCCCC2)c2ccccn2)CC1,79.17,Rattus norvegicus
2797,CHEMBL2325638,O=S(=O)(Nc1ncns1)c1cc(F)c(Oc2cc(F)c(Cl)cc2-c2c...,99.80,Rattus norvegicus
2800,CHEMBL2059859,Cc1ccc(-c2nc3ccccc3[nH]2)cc1NC(=O)c1ccc(OCc2cc...,94.68,Rattus norvegicus


Conflicting SMILES removed


In [7]:
repeated_smiles = labels_df_filtered['SMILES'].value_counts()
repeated_smiles = repeated_smiles[repeated_smiles > 1].index.tolist()
repeated_smiles

if repeated_smiles:
    print(len(repeated_smiles)," duplicate SMILES found:")
    for smiles in repeated_smiles:
        print(smiles)
        display(labels_df_filtered[labels_df_filtered['SMILES'] == smiles])
    print("Removing duplicate SMILES from dataframe")
    labels_df_unique = labels_df_filtered.drop_duplicates(subset=['SMILES'], keep='first')
    print("Done")
    display(labels_df_unique)
else:
    labels_df_unique = labels_df_filtered.copy()
    print("No repeated SMILES found.")

10  duplicate SMILES found:
CC(C)N1C(=O)[C@H](NC(=O)[C@@H](Cc2ccccc2OC(F)(F)F)NC(=O)OC(C)(C)C)CCc2ccccc21


,Drug_ID,SMILES,Y,Species
1471,CHEMBL247828,CC(C)N1C(=O)[C@H](NC(=O)[C@@H](Cc2ccccc2OC(F)(...,99.7,Homo sapiens
2130,CHEMBL247828,CC(C)N1C(=O)[C@H](NC(=O)[C@@H](Cc2ccccc2OC(F)(...,99.7,Rattus norvegicus


O=C(CCCN1CC[Si](O)(c2ccc(Cl)cc2)CC1)c1ccc(F)cc1


,Drug_ID,SMILES,Y,Species
730,CHEMBL2204343,O=C(CCCN1CC[Si](O)(c2ccc(Cl)cc2)CC1)c1ccc(F)cc1,94.56,Homo sapiens
2484,CHEMBL2204343,O=C(CCCN1CC[Si](O)(c2ccc(Cl)cc2)CC1)c1ccc(F)cc1,94.56,Rattus norvegicus


COc1cccc([C@H](O)C2CCN(CCc3ccc(F)cc3)CC2)c1OC


...

,Drug_ID,SMILES,Y,Species
800,CHEMBL2216902,CC(C)c1nc2ccccc2n1-c1nc(N2CCOCC2)c2nc(OC3CN(C4...,78.01,Homo sapiens
2680,CHEMBL2216902,CC(C)c1nc2ccccc2n1-c1nc(N2CCOCC2)c2nc(OC3CN(C4...,78.01,Rattus norvegicus


OCCN1CCN(CCCN2c3ccccc3Sc3ccc(C(F)(F)F)cc32)CC1


,Drug_ID,SMILES,Y,Species
450,CHEMBL726,OCCN1CCN(CCCN2c3ccccc3Sc3ccc(C(F)(F)F)cc32)CC1,98.81,Homo sapiens
2741,CHEMBL726,OCCN1CCN(CCCN2c3ccccc3Sc3ccc(C(F)(F)F)cc32)CC1,98.81,Rattus norvegicus


Removing duplicate SMILES from dataframe
Done


,Drug_ID,SMILES,Y,Species
28,CHEMBL1922660,CCOc1ccc(-c2ccc(Cn3c(CC(C)(C)C(=O)O)c(SC(C)(C)...,99.92,Canis lupus familiaris
30,CHEMBL403225,C[C@H](CO)Nc1nc(SCc2cccc(Cl)c2F)nc2[nH]c(=O)sc12,99.95,Canis lupus familiaris
45,CHEMBL1790041,CNC(=C[N+](=O)[O-])NCCSCc1ccc(CN(C)C)o1,28.95,Canis lupus familiaris
62,CHEMBL2171021,Cc1c(Cl)ccc(OC2CCN(C3CCN(C(=O)c4cccc(S(C)(=O)=...,96.50,Canis lupus familiaris
92,CHEMBL1516,CCCc1nc(C(C)(C)O)c(C(=O)O)n1Cc1ccc(-c2ccccc2-c...,98.81,Canis lupus familiaris
...,...,...,...,...
2791,CHEMBL1580912,O=C(NCCc1ccc(Cl)cc1)c1cccnc1,91.46,Rattus norvegicus
2796,CHEMBL31354,COc1ccccc1N1CCN(CCN(C(=O)C2CCCCC2)c2ccccn2)CC1,79.17,Rattus norvegicus
2797,CHEMBL2325638,O=S(=O)(Nc1ncns1)c1cc(F)c(Oc2cc(F)c(Cl)cc2-c2c...,99.80,Rattus norvegicus
2800,CHEMBL2059859,Cc1ccc(-c2nc3ccccc3[nH]2)cc1NC(=O)c1ccc(OCc2cc...,94.68,Rattus norvegicus


## Save Cleaned Dataset

In [11]:
labels_df_unique.to_csv(f"../data/{dataset}_cleaned.csv")

In [18]:
from ChemicalDice import smiles_to_embeddings

# Generate embeddings from CSV
CDI_embeddings = smiles_to_embeddings.collect_features_from_csv(
    filepath=f"../data/{dataset}_cleaned.csv",
    convert_to_canonical=False
)

CDI_embeddings

All SMILES are valid.
Saved canonical SMILES to temp file: /tmp/tmp2o4p8hj5.csv
Sent /tmp/tmp2o4p8hj5.csv. Receiving stream...


100%|██████████| 35/35 [00:12<00:00,  2.81batch/s]


Stream finished. Concatenating batches...


,SMILES,CDI1,CDI2,CDI3,CDI4,CDI5,CDI6,CDI7,CDI8,CDI9,...,CDI8183,CDI8184,CDI8185,CDI8186,CDI8187,CDI8188,CDI8189,CDI8190,CDI8191,CDI8192
0,CCOc1ccc(-c2ccc(Cn3c(CC(C)(C)C(=O)O)c(SC(C)(C)...,-0.112020,7.800903,0.348821,0.146076,0.351550,0.333945,0.248491,-0.358052,-0.259561,...,0.013584,-0.215027,-0.593232,-0.251186,6.750084,0.082696,-0.179041,0.122496,-0.065559,0.063309
1,C[C@H](CO)Nc1nc(SCc2cccc(Cl)c2F)nc2[nH]c(=O)sc12,-0.134578,7.678442,0.507060,0.361889,0.489886,0.490634,0.326738,-0.439593,-0.443664,...,0.110242,-0.290491,-0.718691,-0.278855,6.935112,0.172814,-0.184790,0.233132,-0.138361,0.116665
2,CNC(=C[N+](=O)[O-])NCCSCc1ccc(CN(C)C)o1,-0.109536,7.664749,0.540561,0.378041,0.510999,0.496168,0.359051,-0.443771,-0.475664,...,0.097660,-0.279299,-0.707880,-0.284046,6.904514,0.124977,-0.166997,0.236739,-0.165415,0.101713
3,Cc1c(Cl)ccc(OC2CCN(C3CCN(C(=O)c4cccc(S(C)(=O)=...,-0.112201,7.737064,0.429881,0.241949,0.405471,0.388015,0.326015,-0.365053,-0.381815,...,0.031259,-0.247952,-0.631331,-0.248380,6.784470,0.083950,-0.162076,0.170783,-0.111983,0.079092
4,CCCc1nc(C(C)(C)O)c(C(=O)O)n1Cc1ccc(-c2ccccc2-c...,-0.129556,7.730204,0.443079,0.269246,0.420570,0.432281,0.310673,-0.387100,-0.363025,...,0.075457,-0.250846,-0.674552,-0.233234,6.841152,0.125219,-0.193701,0.172335,-0.109051,0.096196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1101,O=C(NCCc1ccc(Cl)cc1)c1cccnc1,-0.123465,7.665267,0.535011,0.385863,0.514319,0.519569,0.339115,-0.460828,-0.463383,...,0.129401,-0.308271,-0.743065,-0.288260,6.966096,0.182305,-0.178383,0.258043,-0.157168,0.115244
1102,COc1ccccc1N1CCN(CCN(C(=O)C2CCCCC2)c2ccccn2)CC1,-0.104675,7.688484,0.481717,0.298994,0.462310,0.479108,0.361860,-0.389153,-0.417746,...,0.078062,-0.258481,-0.676209,-0.250144,6.857898,0.097341,-0.183338,0.177266,-0.152129,0.079398
1103,O=S(=O)(Nc1ncns1)c1cc(F)c(Oc2cc(F)c(Cl)cc2-c2c...,-0.139389,7.717937,0.463949,0.289015,0.452213,0.421328,0.308704,-0.418084,-0.405048,...,0.076613,-0.294134,-0.711747,-0.254912,6.882516,0.180593,-0.187847,0.214960,-0.094764,0.130814
1104,Cc1ccc(-c2nc3ccccc3[nH]2)cc1NC(=O)c1ccc(OCc2cc...,-0.123838,7.725387,0.427848,0.287655,0.424018,0.456291,0.287894,-0.430993,-0.342827,...,0.090751,-0.273563,-0.693734,-0.252290,6.921553,0.184151,-0.190561,0.201703,-0.119075,0.111428


## Data Preparation for Modeling

In [19]:
labels_df_unique = labels_df_unique[labels_df_unique['SMILES'].isin(CDI_embeddings['SMILES'])]

In [20]:
X = CDI_embeddings.drop(columns=["SMILES"])

y = labels_df_unique[target_column_name]

## Target Variable Transformation 

In [21]:

# Set the desired transformation method for the target variable 'y'.
# Options:
#   'yeo-johnson' -> Handles positive, zero, and negative values. (Recommended)
#   'log'         -> Use log(1+y) transform. Only works if all y > -1.
#   'none'        -> No transformation is applied.
TARGET_TRANSFORMATION = 'none'

y_orignal_copy= y.copy()
np.random.seed(42)

pt = None # Initialize PowerTransformer object
print(f"--- Target Variable Transformation Mode: {TARGET_TRANSFORMATION.upper()} ---")

if TARGET_TRANSFORMATION == 'yeo-johnson':
    print("Applying Yeo-Johnson transformation to the target variable.")
    pt = PowerTransformer(method='yeo-johnson')
    y_transformed = pt.fit_transform(y.values.reshape(-1, 1))
    y = pd.Series(y_transformed.flatten(), index=y.index)
    print("Transformation complete.")

elif TARGET_TRANSFORMATION == 'log':
    if (y <= -1).any():
        print("Error: Target variable contains values <= -1, which is invalid for log(1+y).")
        print("Please use 'yeo-johnson' or clean the data. Aborting transformation.")
        TARGET_TRANSFORMATION = 'none' # Revert to 'none' to avoid errors
    else:
        print("Applying log1p transformation to the target variable.")
        y = np.log1p(y)
        print("Transformation complete.")

else: # 'none'
    print("No transformation will be applied to the target variable.")


--- Target Variable Transformation Mode: NONE ---
No transformation will be applied to the target variable.


## Model Training and Evaluation

In [ ]:

os.makedirs(  f"../results/{dataset}_results", exist_ok=True)
descriptor_name = "CDI"

result_file = f"../results/{dataset}_results/"+descriptor_name+'_model_metrics.csv'


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models =  {
  "Gradient Boosting Regressor": GradientBoostingRegressor(),
  }
# models =  {
#   "Linear Regression": LinearRegression(),
#   "Ridge Regression": Ridge(),
#   "Lasso Regression": Lasso(),
#   "ElasticNet": ElasticNet(),
#   "Support Vector Regressor": SVR(),
#   "Random Forest Regressor": RandomForestRegressor(),#time taking models, run only  if recources are not limited
#   "Gradient Boosting Regressor": GradientBoostingRegressor(),
#   "Extra Trees Regressor": ExtraTreesRegressor(),
#   "XGBoost Regressor": XGBRegressor(),
#   "LightGBM Regressor": LGBMRegressor(),
#   "CatBoost Regressor": CatBoostRegressor(verbose=0) #time taking models, run only  if recources are not limited
#   }


print(f"Training set size: {X_train_scaled.shape[0]} samples, {X_train_scaled.shape[1]} features")
print(f"Testing set size: {X_test_scaled.shape[0]} samples, {X_test_scaled.shape[1]} features")
results = {}
for name, model in models.items():
    print(f"Training model: {name}")

    # --- Time tracking ---
    start_time = time.time()


    # --- Training ---
    model.fit(X_train_scaled, y_train)

    # # --- Predictions ---
    y_pred_test = model.predict(X_test_scaled)
    y_pred_train = model.predict(X_train_scaled)
    elapsed_time = round(time.time() - start_time, 4)

    # <<< TACKLE INFINITY VALUES >>>
    if TARGET_TRANSFORMATION != 'none':
        train_range = y_train.max() - y_train.min()
        safe_min = y_train.min() - 0.2 * train_range
        safe_max = y_train.max() + 0.2 * train_range
        if np.any(y_pred_train > safe_max) or np.any(y_pred_train < safe_min):
            print(f"    ⚠️ WARNING: Model '{name}' produced extreme predictions on TRAINING data.")
        if np.any(y_pred_test > safe_max) or np.any(y_pred_test < safe_min):
            print(f"    ⚠️ WARNING: Model '{name}' produced extreme predictions on TEST data.")
        y_pred_train = np.clip(y_pred_train, safe_min, safe_max)
        y_pred_test = np.clip(y_pred_test, safe_min, safe_max)

    # Inverse transform
    if TARGET_TRANSFORMATION == 'yeo-johnson':
        y_test_orig = pt.inverse_transform(y_test.values.reshape(-1, 1)).flatten()
        y_train_orig = pt.inverse_transform(y_train.values.reshape(-1, 1)).flatten()
        y_pred_test_orig = pt.inverse_transform(y_pred_test.reshape(-1, 1)).flatten()
        y_pred_train_orig = pt.inverse_transform(y_pred_train.reshape(-1, 1)).flatten()
    elif TARGET_TRANSFORMATION == 'log':
        y_test_orig, y_train_orig = np.expm1(y_test), np.expm1(y_train)
        y_pred_test_orig, y_pred_train_orig = np.expm1(y_pred_test), np.expm1(y_pred_train)
    else: # 'none'
        y_test_orig, y_train_orig = y_test, y_train
        y_pred_test_orig, y_pred_train_orig = y_pred_test, y_pred_train


    # Store metrics calculated on the original scale
    results[name] = {
        'Time Taken (s)': elapsed_time,
        'Test R2 Score': r2_score(y_test_orig, y_pred_test_orig),
        'Test MAE': mean_absolute_error(y_test_orig, y_pred_test_orig),
        'Test RMSE': np.sqrt(mean_squared_error(y_test_orig, y_pred_test_orig)),
        'Test MSE': mean_squared_error(y_test_orig, y_pred_test_orig),
        'Train R2 Score': r2_score(y_train_orig, y_pred_train_orig),
        'Train MAE': mean_absolute_error(y_train_orig, y_pred_train_orig),
        'Train RMSE': np.sqrt(mean_squared_error(y_train_orig, y_pred_train_orig)),
        'Train MSE': mean_squared_error(y_train_orig, y_pred_train_orig),
    }
# --- Save and display results ---
results_df = pd.DataFrame.from_dict(results, orient='index')
results_df.to_csv(result_file)
display(descriptor_name)
display(results_df)

Training set size: 884 samples, 8192 features
Testing set size: 222 samples, 8192 features
Training model: Linear Regression
Training model: Ridge Regression


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=6.5711e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


Training model: Lasso Regression
Training model: ElasticNet
Training model: Support Vector Regressor
Training model: Random Forest Regressor
Training model: Gradient Boosting Regressor
Training model: Extra Trees Regressor
Training model: XGBoost Regressor
Training model: LightGBM Regressor


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.377597 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2088960
[LightGBM] [Info] Number of data points in the train set: 884, number of used features: 8192
[LightGBM] [Info] Start training from score 86.887704


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Training model: CatBoost Regressor


'CDI'

,Time Taken (s),Test R2 Score,Test MAE,Test RMSE,Test MSE,Train R2 Score,Train MAE,Train RMSE,Train MSE
Linear Regression,1.2371,-0.592401,14.766258,20.611255,424.823833,0.834375,5.456710e+00,7.100819e+00,5.042162e+01
Ridge Regression,0.3444,-0.624934,15.019173,20.820737,433.503104,0.916325,3.842405e+00,5.047130e+00,2.547352e+01
Lasso Regression,0.3036,0.173560,10.822790,14.848544,220.479269,0.313540,1.040573e+01,1.445616e+01,2.089807e+02
ElasticNet,1.0090,0.186879,10.592384,14.728406,216.925930,0.350068,1.013703e+01,1.406628e+01,1.978603e+02
Support Vector Regressor,16.1657,0.005016,9.512508,16.292442,265.443675,0.080756,9.622534e+00,1.672865e+01,2.798477e+02
Random Forest Regressor,1260.0504,0.301302,9.282026,13.652838,186.399983,0.907536,3.659853e+00,5.305579e+00,2.814917e+01
Gradient Boosting Regressor,418.8366,0.301074,8.998697,13.655063,186.460751,0.924175,3.680859e+00,4.804548e+00,2.308368e+01
Extra Trees Regressor,279.1469,0.318238,9.287458,13.486357,181.881837,1.000000,9.124124e-14,1.088790e-13,1.185464e-26
XGBoost Regressor,256.1845,0.206126,9.426617,14.553053,211.791360,1.000000,6.352079e-03,8.645462e-03,7.474402e-05
LightGBM Regressor,123.5090,0.294402,9.035139,13.720086,188.240757,0.994496,6.287625e-01,1.294417e+00,1.675514e+00
